# 08 · 實戰專案 · 接軌真實世界

把前七課全部組起來：一個**迷你研究助理**——會查資料(RAG)、會算數、記得住對話、會多步推理。最後一個選修 sidebar，只改 `chat()` 一個函式，就把本地 Qwen 換成免費 API。

## 1. 載入所有零件

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes
%pip install -q sentence-transformers

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

In [ ]:
# 內嵌知識庫：每張卡片一段事實短文（真實世界資料，零下載）。
KB = [
    "玉山是台灣最高峰，主峰海拔 3952 公尺，位於玉山國家公園，橫跨南投、嘉義、高雄。",
    "台灣最長的河流是濁水溪，全長約 186.6 公里，發源於合歡山，向西流入台灣海峽。",
    "日月潭是台灣最大的天然湖泊，位於南投縣魚池鄉，潭面海拔約 748 公尺。",
    "台灣本島面積約 36,197 平方公里，是世界第 38 大島嶼，南北長約 394 公里。",
    "台北101 樓高 508 公尺、地上 101 層，2004 年完工，曾是世界第一高樓直到 2010 年。",
    "台灣高鐵全長約 350 公里，連接南港與左營，最高營運時速 300 公里，2007 年通車。",
    "台灣的氣候北部為副熱帶、南部為熱帶，每年 5 到 6 月有梅雨季，夏秋多颱風。",
    "澎湖群島由約 90 座島嶼組成，位於台灣海峽中央，以玄武岩柱狀節理地形聞名。",
    "台灣原住民族目前官方認定有 16 族，人口約佔總人口的 2.5%，阿美族是人數最多的一族。",
    "蘭嶼位於台東外海，是達悟族（雅美族）的居住地，以拼板舟與飛魚文化著稱。",
]

In [ ]:
from datetime import datetime, timezone, timedelta


def get_current_time() -> str:
    return datetime.now(timezone(timedelta(hours=8))).strftime("%Y-%m-%d %H:%M:%S")


def calculator(expression: str) -> str:
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression or ""):
        return "錯誤：只接受數字與 + - * / ( ) . %"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"錯誤：{e}"


TOOLS = {"get_current_time": get_current_time, "calculator": calculator}
REACT_SYSTEM = """你是會用工具、一步步推理的助手。工具：
- get_current_time()：現在台北時間，無參數。
- calculator(expression)：算式，例 {"expression": "15-9"}。
格式（一次一步）：
Thought: 推理
Action: 工具名
Action Input: JSON
我會回 Observation。足夠時改用：
Thought: 推理
Final Answer: 答案"""


def parse_action(text):
    m = re.search(r"Action:\s*([A-Za-z_]\w*)", text)
    if not m:
        return None, None
    args, ma = {}, re.search(r"Action Input:\s*(\{.*\})", text, re.DOTALL)
    if ma:
        try:
            args = json.loads(ma.group(1))
        except Exception:
            args = {}
    return m.group(1), args


def run_react(question, max_steps=5, verbose=False):
    pad = ""
    for _ in range(max_steps):
        msg = [{"role": "system", "content": REACT_SYSTEM},
               {"role": "user", "content": f"問題：{question}\n\n{pad}"}]
        r = chat(msg, max_new_tokens=256).split("Observation")[0].strip()
        if verbose:
            print(r)
        if "Final Answer:" in r:
            return r.split("Final Answer:")[-1].strip()
        name, args = parse_action(r)
        obs = "（沒解析到 Action）" if not name else (
            TOOLS[name](**args) if name in TOOLS else f"沒有 {name} 這支工具")
        pad += f"{r}\nObservation: {obs}\n"
    return "（達到最大步數）"

## 2. 把 RAG 檢索也加進工具箱

研究助理 = 第 04 課路由 + 第 06 課 RAG。

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
kb_vecs = embedder.encode(KB, normalize_embeddings=True)


def retrieve(query: str) -> str:
    q = embedder.encode([query], normalize_embeddings=True)[0]
    top = np.argsort(-(kb_vecs @ q))[:2]
    return " | ".join(KB[i] for i in top)


# 把 retrieve 註冊進工具箱 + prompt（沿用 EXECUTOR 的 TOOLS / REACT_SYSTEM）
TOOLS["retrieve"] = retrieve
globals()["REACT_SYSTEM"] = REACT_SYSTEM.replace(
    "格式（一次一步）",
    '- retrieve(query)：查台灣常識知識庫，例 {"query": "玉山多高"}。\n格式（一次一步）')

## 3. 跑一個「查 + 算」綜合問題

In [ ]:
print(run_react("玉山的海拔（公尺）減去台北101 的高度（公尺）是多少？", verbose=True))
# agent 會：retrieve 查玉山 → retrieve 查台北101 → calculator 相減 → Final Answer

## 4. 選修 sidebar：接軌真實世界（免費 API）

因為一切都透過 `chat()`，要換更強的模型**只改這一個函式**，agent 其餘程式一行不動。

- **Gemini 2.5 Flash**：1,500 req/天、免卡、支援 function calling。
- ⚠️ 各自申請自己的 key，用 Colab secrets / 環境變數帶入，**別寫死在 notebook**。

In [ ]:
# 取消註解即可改用 Gemini（需先 %pip install -q google-generativeai 並設好金鑰）
# import os, google.generativeai as genai
# genai.configure(api_key=os.environ["GEMINI_API_KEY"])
# _g = genai.GenerativeModel("gemini-2.5-flash")
#
# def chat(messages, max_new_tokens=512, temperature=0.0):
#     """同樣的介面：messages 進、文字出。把多輪訊息攤平成一段 prompt。"""
#     prompt = "\n".join(f'{m["role"]}: {m["content"]}' for m in messages)
#     resp = _g.generate_content(
#         prompt, generation_config={"temperature": temperature,
#                                    "max_output_tokens": max_new_tokens})
#     return resp.text.strip()
#
# 換上去後，前面所有 run_react / orchestrate 完全不用改，立刻感覺 tool calling 變穩。
print("把上面那段取消註解、設好 GEMINI_API_KEY，就能一鍵換模型。")

## 終點，也是起點

你把**工具、ReAct、路由、記憶、RAG、多代理**全部組成了一個能用的 agent。

從 `ml` 的第一個分類器，到 `llm` 從零刻出的迷你 GPT，再到這條軌道**手刻一個會用工具、會做事的 agent**——你走完了「經典 ML → 深度學習 → 從零打造 LLM → AI Agent」整條學習弧線。

真實世界的 agent 框架（LangGraph、AutoGen…）只是把你親手寫過的這些零件做得更工整。**底層原理，你已經全部刻過一遍了。**

> 🎓 恭喜走完整條 AI/ML 學習線。下一步：拿這套骨架，去打造一個真正解決你自己問題的 agent。